# F1 CFD Worker — Kaggle

This notebook is the Kaggle equivalent of the Colab CFD worker.
It connects to Google Drive via **rclone** instead of the native Colab mount.

## One-time Kaggle setup

1. On your **Mac**, install rclone:  
   `brew install rclone`

2. Configure rclone for Google Drive:  
   `rclone config`  
   → New remote → Name: `gdrive` → Type: `drive` → follow browser auth prompts

3. Copy the rclone config to clipboard:  
   `cat ~/.config/rclone/rclone.conf | pbcopy`

4. In Kaggle:  
   **Your Profile → Settings → Secrets → Add New Secret**  
   Name: `RCLONE_CONF`  
   Value: paste the config

5. Enable **Internet** and **GPU** in this notebook's Settings panel.

**Keep this tab open during your session.** Kaggle sessions last up to 9 hours.

In [ ]:
# Cell 1: Install rclone and write config from Kaggle secret
import subprocess, os
from pathlib import Path

def run(cmd, **kwargs):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kwargs)
    if r.returncode != 0:
        print('STDERR:', r.stderr[-2000:])
    return r.stdout, r.stderr, r.returncode

# Install rclone
print('Installing rclone...')
run('curl -s https://rclone.org/install.sh | bash')

# Write rclone config from Kaggle secret
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    rclone_conf = secrets.get_secret('RCLONE_CONF')
    conf_dir = Path('/root/.config/rclone')
    conf_dir.mkdir(parents=True, exist_ok=True)
    (conf_dir / 'rclone.conf').write_text(rclone_conf)
    print('rclone config written from secret.')
except Exception as e:
    print(f'ERROR: Could not load RCLONE_CONF secret: {e}')
    print('Make sure you added the secret in Kaggle Settings.')
    raise

# Test connection
out, err, rc = run('rclone lsd gdrive:f1-opt')
if rc == 0:
    print('Google Drive connected! f1-opt contents:', out.strip() or '(empty)')
else:
    print('Drive connection test failed:', err)

In [ ]:
# Cell 2: Install OpenFOAM
print('Adding OpenFOAM repo...')
run('curl -s https://dl.openfoam.com/add-debian-repo.sh | bash')
run('apt-get update -qq')
print('Installing OpenFOAM (takes a few minutes)...')
run('apt-get install -y -qq openfoam2312')
print('Done.')

In [ ]:
# Cell 3: Setup paths
import json, time, shutil, datetime
from pathlib import Path

# Local mirror of the Drive queue/results folders
LOCAL_MIRROR = Path('/tmp/f1-opt')
LOCAL_QUEUE   = LOCAL_MIRROR / 'queue'
LOCAL_RESULTS = LOCAL_MIRROR / 'results'
WORK_DIR      = Path('/tmp/cfd_work')
WORKER_ID     = 'kaggle'
OF_BASHRC     = '/usr/lib/openfoam/openfoam2312/etc/bashrc'

for d in (LOCAL_QUEUE, LOCAL_RESULTS, WORK_DIR):
    d.mkdir(parents=True, exist_ok=True)

REMOTE = 'gdrive:f1-opt'

def sync_from_drive():
    """Pull latest queue from Drive to local mirror."""
    run(f'rclone sync {REMOTE}/queue {LOCAL_QUEUE} --transfers 4 -q')

def push_result(result_file: Path):
    """Push a result JSON to Drive results folder."""
    run(f'rclone copy {result_file} {REMOTE}/results -q')

def of_run(cmd, cwd=None):
    full = f'bash -c "source {OF_BASHRC} && {cmd}"'
    r = subprocess.run(full, shell=True, capture_output=True, text=True, cwd=cwd)
    return r.stdout, r.stderr, r.returncode

print(f'Worker {WORKER_ID} ready.')

In [ ]:
# Cell 4: Case writer and results parser
# (identical logic to Colab worker — copied here for self-contained operation)
import math

def write_case(case_dir: Path, stl_path: Path):
    (case_dir / '0').mkdir(parents=True, exist_ok=True)
    (case_dir / 'constant' / 'triSurface').mkdir(parents=True, exist_ok=True)
    (case_dir / 'system').mkdir(parents=True, exist_ok=True)
    shutil.copy2(stl_path, case_dir / 'constant' / 'triSurface' / 'car.stl')

    (case_dir / 'system' / 'blockMeshDict').write_text("""\
FoamFile { version 2.0; format ascii; class dictionary; object blockMeshDict; }
convertToMeters 1;
vertices
(
    (-20 -10  0)  // 0
    ( 40 -10  0)  // 1
    ( 40  10  0)  // 2
    (-20  10  0)  // 3
    (-20 -10  8)  // 4
    ( 40 -10  8)  // 5
    ( 40  10  8)  // 6
    (-20  10  8)  // 7
);
blocks ( hex (0 1 2 3 4 5 6 7) (60 20 8) simpleGrading (1 1 1) );
edges ();
boundary
(
    inlet  { type patch; faces ((0 4 7 3)); }
    outlet { type patch; faces ((1 2 6 5)); }
    sides  { type symmetryPlane; faces ((0 1 5 4)(3 7 6 2)); }
    top    { type symmetryPlane; faces ((4 5 6 7)); }
    ground { type wall; faces ((0 3 2 1)); }
);
""")

    (case_dir / 'system' / 'snappyHexMeshDict').write_text("""\
FoamFile { version 2.0; format ascii; class dictionary; object snappyHexMeshDict; }
castellatedMesh true;
snap            true;
addLayers       false;
geometry { car.stl { type triSurfaceMesh; name car; } }
castellatedMeshControls
{
    maxLocalCells 1000000; maxGlobalCells 2000000; minRefinementCells 0;
    nCellsBetweenLevels 3; features ();
    refinementSurfaces { car { level (4 5); patchInfo { type wall; } } }
    refinementRegions {}
    locationInMesh (5 0 1);
    allowFreeStandingZoneFaces false;
}
snapControls { nSmoothPatch 3; tolerance 4.0; nSolveIter 30; nRelaxIter 5; nFeatureSnapIter 10; implicitFeatureSnap false; explicitFeatureSnap true; multiRegionFeatureSnap false; }
addLayersControls { relativeSizes true; layers {} expansionRatio 1.0; finalLayerThickness 0.3; minThickness 0.1; }
meshQualityControls { maxNonOrtho 65; maxBoundarySkewness 20; maxInternalSkewness 4; maxConcave 80; minVol 1e-13; minTetQuality -1e30; minArea -1; minTwist 0.01; minDeterminant 0.001; minFaceWeight 0.05; minVolRatio 0.01; minTriangleTwist -1; nSmoothScale 4; errorReduction 0.75; }
writeFlags ( scalarLevels layerSets layerFields );
mergeTolerance 1e-6;
""")

    (case_dir / 'constant' / 'transportProperties').write_text("""\
FoamFile { version 2.0; format ascii; class dictionary; object transportProperties; }
transportModel Newtonian;
nu  1.5e-05;
""")
    (case_dir / 'constant' / 'turbulenceProperties').write_text("""\
FoamFile { version 2.0; format ascii; class dictionary; object turbulenceProperties; }
simulationType RAS;
RAS { RASModel kOmegaSST; turbulence on; printCoeffs on; }
""")

    U = 60.0
    k_val = 1.5 * (U * 0.01) ** 2
    omega_val = math.sqrt(k_val) / (0.09**0.25 * 0.1)

    (case_dir / '0' / 'U').write_text(f"FoamFile {{ version 2.0; format ascii; class volVectorField; object U; }}\ndimensions [0 1 -1 0 0 0 0];\ninternalField uniform ({U} 0 0);\nboundaryField\n{{\n    inlet   {{ type fixedValue; value uniform ({U} 0 0); }}\n    outlet  {{ type zeroGradient; }}\n    sides   {{ type symmetryPlane; }}\n    top     {{ type symmetryPlane; }}\n    ground  {{ type fixedValue; value uniform ({U} 0 0); }}\n    car     {{ type noSlip; }}\n}}\n")
    (case_dir / '0' / 'p').write_text("FoamFile { version 2.0; format ascii; class volScalarField; object p; }\ndimensions [0 2 -2 0 0 0 0];\ninternalField uniform 0;\nboundaryField\n{\n    inlet   { type zeroGradient; }\n    outlet  { type fixedValue; value uniform 0; }\n    sides   { type symmetryPlane; }\n    top     { type symmetryPlane; }\n    ground  { type zeroGradient; }\n    car     { type zeroGradient; }\n}\n")
    (case_dir / '0' / 'k').write_text(f"FoamFile {{ version 2.0; format ascii; class volScalarField; object k; }}\ndimensions [0 2 -2 0 0 0 0];\ninternalField uniform {k_val:.4f};\nboundaryField\n{{\n    inlet   {{ type fixedValue; value uniform {k_val:.4f}; }}\n    outlet  {{ type zeroGradient; }}\n    sides   {{ type symmetryPlane; }}\n    top     {{ type symmetryPlane; }}\n    ground  {{ type zeroGradient; }}\n    car     {{ type kqRWallFunction; value uniform {k_val:.4f}; }}\n}}\n")
    (case_dir / '0' / 'omega').write_text(f"FoamFile {{ version 2.0; format ascii; class volScalarField; object omega; }}\ndimensions [0 0 -1 0 0 0 0];\ninternalField uniform {omega_val:.2f};\nboundaryField\n{{\n    inlet   {{ type fixedValue; value uniform {omega_val:.2f}; }}\n    outlet  {{ type zeroGradient; }}\n    sides   {{ type symmetryPlane; }}\n    top     {{ type symmetryPlane; }}\n    ground  {{ type zeroGradient; }}\n    car     {{ type omegaWallFunction; value uniform {omega_val:.2f}; }}\n}}\n")
    (case_dir / '0' / 'nut').write_text("FoamFile { version 2.0; format ascii; class volScalarField; object nut; }\ndimensions [0 2 -1 0 0 0 0];\ninternalField uniform 0;\nboundaryField\n{\n    inlet   { type calculated; value uniform 0; }\n    outlet  { type calculated; value uniform 0; }\n    sides   { type symmetryPlane; }\n    top     { type symmetryPlane; }\n    ground  { type nutkWallFunction; value uniform 0; }\n    car     { type nutkWallFunction; value uniform 0; }\n}\n")

    (case_dir / 'system' / 'controlDict').write_text("""\
FoamFile { version 2.0; format ascii; class dictionary; object controlDict; }
application     simpleFoam;
startFrom       startTime;
startTime       0;
stopAt          endTime;
endTime         300;
deltaT          1;
writeControl    timeStep;
writeInterval   300;
purgeWrite      1;
writeFormat     ascii;
writePrecision  6;
writeCompression off;
timeFormat      general;
timePrecision   6;
runTimeModifiable true;
functions
{
    forceCoeffs
    {
        type            forceCoeffs;
        libs            (forces);
        writeControl    timeStep;
        writeInterval   10;
        patches         (car);
        rho             rhoInf;
        rhoInf          1.225;
        liftDir         (0 0 1);
        dragDir         (1 0 0);
        CofR            (2.5 0 0.3);
        pitchAxis       (0 1 0);
        magUInf         60;
        lRef            5.6;
        Aref            1.5;
    }
}
""")
    (case_dir / 'system' / 'fvSchemes').write_text("""\
FoamFile { version 2.0; format ascii; class dictionary; object fvSchemes; }
ddtSchemes   { default steadyState; }
gradSchemes  { default Gauss linear; }
divSchemes
{
    default         none;
    div(phi,U)      bounded Gauss linearUpwind grad(U);
    div(phi,k)      bounded Gauss linearUpwind grad(k);
    div(phi,omega)  bounded Gauss linearUpwind grad(omega);
    div((nuEff*dev(T(grad(U))))) Gauss linear;
}
laplacianSchemes { default Gauss linear corrected; }
interpolationSchemes { default linear; }
snGradSchemes { default corrected; }
""")
    (case_dir / 'system' / 'fvSolution').write_text("""\
FoamFile { version 2.0; format ascii; class dictionary; object fvSolution; }
solvers
{
    p   { solver GAMG; smoother GaussSeidel; tolerance 1e-6; relTol 0.1; }
    U   { solver smoothSolver; smoother symGaussSeidel; tolerance 1e-6; relTol 0.1; }
    k   { solver smoothSolver; smoother symGaussSeidel; tolerance 1e-6; relTol 0.1; }
    omega { solver smoothSolver; smoother symGaussSeidel; tolerance 1e-6; relTol 0.1; }
}
SIMPLE
{
    nNonOrthogonalCorrectors 1;
    consistent yes;
    residualControl { p 1e-4; U 1e-4; k 1e-4; omega 1e-4; }
}
relaxationFactors { fields { p 0.3; } equations { U 0.7; k 0.7; omega 0.7; } }
""")


def parse_force_coeffs(case_dir: Path) -> dict:
    coeff_file = case_dir / 'postProcessing' / 'forceCoeffs' / '0' / 'coefficient.dat'
    if not coeff_file.exists():
        raise FileNotFoundError(f'No coefficient file at {coeff_file}')
    last = None
    with coeff_file.open() as f:
        for line in f:
            line = line.strip()
            if line.startswith('#') or not line:
                continue
            parts = line.split()
            if len(parts) >= 4:
                try:
                    last = {'cm': float(parts[1]), 'cd': float(parts[2]), 'cl': float(parts[3])}
                except ValueError:
                    pass
    if last is None:
        raise ValueError('No valid force coefficient data found')
    cd = max(last['cd'], 0.001)
    last['score'] = last['cl'] / cd
    return last


print('Functions ready.')

In [ ]:
# Cell 5: Main CFD worker loop
import subprocess

POLL_INTERVAL = 30

print(f'CFD Worker ({WORKER_ID}) started at {datetime.datetime.now().strftime("%H:%M:%S")}')
print('Syncing from Drive and watching for jobs...\n')

processed = set()

while True:
    # Sync queue from Drive
    sync_from_drive()

    job_files = sorted(LOCAL_QUEUE.glob('*.json'))

    for job_file in job_files:
        job_id = job_file.stem
        if job_id in processed:
            continue

        stl_file = LOCAL_QUEUE / f'{job_id}.stl'
        if not stl_file.exists():
            continue

        print(f'[{datetime.datetime.now().strftime("%H:%M:%S")}] Processing: {job_id}')
        processed.add(job_id)

        case_dir = WORK_DIR / job_id
        if case_dir.exists():
            shutil.rmtree(case_dir)

        try:
            write_case(case_dir, stl_file)

            print('  blockMesh...')
            out, err, rc = of_run('blockMesh', cwd=case_dir)
            if rc != 0:
                raise RuntimeError(f'blockMesh failed: {err[-500:]}')

            print('  snappyHexMesh...')
            out, err, rc = of_run('snappyHexMesh -overwrite', cwd=case_dir)
            if rc != 0:
                raise RuntimeError(f'snappyHexMesh failed: {err[-500:]}')

            print('  simpleFoam...')
            out, err, rc = of_run('simpleFoam', cwd=case_dir)
            if rc != 0:
                raise RuntimeError(f'simpleFoam failed: {err[-500:]}')

            results = parse_force_coeffs(case_dir)
            results['job_id'] = job_id
            results['worker'] = WORKER_ID
            results['timestamp'] = datetime.datetime.now().isoformat()

            result_path = LOCAL_RESULTS / f'{job_id}.json'
            result_path.write_text(json.dumps(results, indent=2))
            push_result(result_path)

            print(f'  CL={results["cl"]:.4f}  CD={results["cd"]:.4f}  L/D={results["score"]:.3f}')

        except Exception as e:
            print(f'  ERROR: {e}')
            err_result = {
                'job_id': job_id, 'worker': WORKER_ID,
                'error': str(e), 'cl': 0.0, 'cd': 1.0, 'score': 0.0,
                'timestamp': datetime.datetime.now().isoformat(),
            }
            rp = LOCAL_RESULTS / f'{job_id}.json'
            rp.write_text(json.dumps(err_result, indent=2))
            push_result(rp)

        finally:
            if case_dir.exists():
                shutil.rmtree(case_dir)

    time.sleep(POLL_INTERVAL)